# 07 — RAG agéntico: el juez selecciona los chunks (vía Ollama)

Segundo diseño de RAG agéntico, el que pidió el usuario: el sistema SOLO califica los chunks del RAG híbrido y selecciona los útiles con un juez. Sin graph_query, sin calculator.

Por pregunta: (1) retrieval híbrido top-10 (BM25 Okapi + MiniLM, RRF k=60); (2) UN llamado al juez con los 10 chunks numerados, que devuelve un veredicto por chunk (UTIL / NO UTIL) con su razón corta — máximo 3 reintentos si los veredictos no se pueden parsear; si a los 3 sigue sin veredictos válidos, la pregunta se abstiene sin llamar al lector; (3) el lector responde ÚNICAMENTE con los chunks seleccionados (mismo lector, mismo condense, mismo prompt que `rag_baseline/answer.py`); si el juez no selecciona nada, abstención honesta.

**Juez y lector vía ChatOllama** contra el servidor local (`JUDGE_OLLAMA_MODEL`, default `qwen2.5:7b`; `qwen2.5:1.5b` para paridad con el run en CPU). El código de referencia vive en el repo (`agentic_rag/judge_rerank.py`: SYSTEM y parseo de veredictos; `agentic_rag/run_judge.py`: pipeline completo en CPU; `agentic_rag/score_judge.py`: scorer con diagnóstico del juez).

**El intento anterior queda como negativo documentado:** el agente ReAct con LangChain + LangGraph y 3 herramientas sacó 2/12 en CPU y nunca usó `vector_search` ni `calculator` (24/24 llamadas fueron `graph_query`): midió un agente solo-grafo, no un RAG agéntico. Ese run no se toca (`agentic_rag/results_agentic.md`).

**Prerrequisitos:** notebook 01 (chunks). El retrieval híbrido top-10 se genera en este mismo notebook.

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros ---
JUDGE_OLLAMA_MODEL = "qwen2.5:7b"   # juez y lector: 7b (default, el que muestra lo agentico)
                                     # o qwen2.5:1.5b para paridad con el run en CPU
K = 10
RETRIEVAL_K10 = REPO + "/rag_baseline/retrieval_hybrid_k10.json"
TRACE_OUT = REPO + "/agentic_rag/run_trace_judge_colab.jsonl"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive o generándolo del repo con `make extract` (`python pipeline/extract.py`).
3. **Solo si no lo subes**, generarlo aquí corriendo `extract.py` (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal.

`triples_seed.json` y `eval_questions.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive, o del repo con `make extract`.")
    print("Si no lo subes, el notebook lo genera con extract.py (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar con extract.py...")
        print("(descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        try:
            import pypdf
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1"], check=True)
        r = subprocess.run([sys.executable, "extract.py"], cwd=REPO + "/pipeline")
        print("extract.py exit:", r.returncode)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 sentence-transformers==6.3.0 numpy>=2.5.5 rank-bm25==0.2.2 langchain-ollama

## Ollama (setup compacto e idempotente)

Este notebook usa modelos intermedios vía **Ollama** corriendo en la propia instancia de Colab (un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook). Las celdas son idempotentes: verifican si el binario existe, si el servidor ya responde y si el modelo ya está bajado antes de hacer nada. La versión completa, con la explicación y las fuentes, está en `00_ollama_setup.ipynb`.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

In [ ]:
%%bash
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
ollama list

In [ ]:
from langchain_ollama import ChatOllama

_llm = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434", temperature=0)
print(_llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("ChatOllama OK contra", JUDGE_OLLAMA_MODEL)

## 1. Retrieval híbrido top-10

Corre `rag_baseline/retrieve_hybrid.py` con `--k 10`: BM25 Okapi + MiniLM, fusión RRF k=60, top-10 por pregunta (el juez necesita más de donde elegir que el top-5 fijo). Determinista: se puede regenerar sin cambiar el resultado.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "retrieve_hybrid.py", "--k", str(K),
                    "--out", RETRIEVAL_K10], cwd=REPO + "/rag_baseline")
print("exit:", r.returncode)

## 2. Juez + lector (ChatOllama)

Un llamado al juez por pregunta con los 10 chunks numerados (misma versión condensada que verá el lector, mismo SYSTEM y mismo parseo de veredictos que `agentic_rag/judge_rerank.py`; máximo 3 reintentos; si no hay veredictos válidos, abstención sin lector). El lector contesta solo con los chunks seleccionados (mismo SYSTEM y `condense()` de `rag_baseline/answer.py`). Reanudable: salta las preguntas cuyo id ya está en la traza (`run_trace_judge_colab.jsonl`, nunca toca la traza del run en CPU).

In [ ]:
import json, sys, time
sys.path.insert(0, REPO + "/agentic_rag")
sys.path.insert(0, REPO + "/rag_baseline")
from judge_rerank import SYSTEM as JUDGE_SYSTEM, parse_verdicts
from answer import SYSTEM as READER_SYSTEM, condense
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

ABSTAIN = "No lo sé."
judge_llm = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434",
                       temperature=0, num_predict=200)
reader_llm = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434",
                        temperature=0, num_predict=100)
print("juez/lector (Ollama):", JUDGE_OLLAMA_MODEL)

def judge_once(question, chunks):
    # chunks: lista de (page, condensed_text). Un llamado, 10 veredictos.
    parts = [f"[{i}] Página {p}\n{c}" for i, (p, c) in enumerate(chunks, start=1)]
    user = (f"Pregunta: {question}\n\nFragmentos del informe anual "
            f"de FEMSA 2024:\n\n" + "\n\n".join(parts))
    return judge_llm.invoke([SystemMessage(content=JUDGE_SYSTEM),
                             HumanMessage(content=user)]).content.strip()

def run_question(item):
    t0 = time.time()
    chunks = [(r["page"], condense(r["text"], item["question"]))
              for r in item["retrieved"]]
    presented = [{"n": i + 1, "page": p, "condensed": c}
                 for i, (p, c) in enumerate(chunks)]
    rec = {"id": item["id"], "level": item["level"], "question": item["question"],
           "decomposition": "judge-rerank",
           "reader_model": f"ollama {JUDGE_OLLAMA_MODEL}",
           "retrieved_pages": [r["page"] for r in item["retrieved"]],
           "retrieval_scores": [r["score"] for r in item["retrieved"]],
           "chunks_presented": presented,
           "judge": {"attempts": 0, "attempts_raw": [], "verdicts": None,
                     "ok": False, "seconds": 0.0},
           "selected_pages": [],
           "reader": {"attempts": 0, "seconds": 0.0, "failed": False, "skipped": False},
           "context": "", "final_answer": ABSTAIN, "seconds": 0.0}
    # Juez: maximo 3 reintentos si no se pueden parsear los veredictos.
    tj = time.time()
    verdicts = None
    for attempt in range(1, 4):
        try:
            raw = judge_once(item["question"], chunks)
        except Exception as e:
            raw = f"<ERROR: {type(e).__name__}: {e}>"
        rec["judge"]["attempts_raw"].append(raw)
        rec["judge"]["attempts"] = attempt
        verdicts = parse_verdicts(raw, len(chunks))
        if verdicts is not None:
            break
    rec["judge"]["seconds"] = round(time.time() - tj, 1)
    if verdicts is None:
        rec["judge"]["ok"] = False
        rec["reader"]["skipped"] = True
        rec["seconds"] = round(time.time() - t0, 1)
        return rec
    rec["judge"]["ok"] = True
    rec["judge"]["verdicts"] = verdicts
    selected = [i for i in range(1, len(chunks) + 1) if verdicts[i]["useful"]]
    rec["selected_pages"] = [chunks[i - 1][0] for i in selected]
    if not selected:
        rec["reader"]["skipped"] = True
        rec["seconds"] = round(time.time() - t0, 1)
        return rec
    ctx = "\n\n".join(f"[Página {chunks[i - 1][0]}]\n{chunks[i - 1][1]}" for i in selected)
    rec["context"] = ctx
    # Lector: maximo 3 reintentos si el llamado falla o viene vacio.
    tr = time.time()
    answer, failed = None, False
    user = (f"Fragmentos del informe anual de FEMSA 2024:\n\n{ctx}\n\n"
            f"Pregunta: {item['question']}")
    for attempt in range(1, 4):
        try:
            ans = reader_llm.invoke([SystemMessage(content=READER_SYSTEM),
                                     HumanMessage(content=user)]).content.strip()
        except Exception as e:
            ans = ""
            print(f"  {item['id']} lector intento {attempt} fallo: {type(e).__name__}: {e}", flush=True)
        if ans:
            answer = ans
            rec["reader"]["attempts"] = attempt
            break
        rec["reader"]["attempts"] = attempt
    else:
        failed = True
    rec["reader"]["seconds"] = round(time.time() - tr, 1)
    rec["reader"]["failed"] = failed
    rec["final_answer"] = ABSTAIN if failed else answer
    rec["seconds"] = round(time.time() - t0, 1)
    return rec

done = set()
try:
    for line in open(TRACE_OUT):
        try:
            done.add(json.loads(line)["id"])
        except (json.JSONDecodeError, KeyError):
            pass
except FileNotFoundError:
    pass
if done:
    print(f"reanudando: {sorted(done)} ya en {TRACE_OUT}")

qs = json.load(open(RETRIEVAL_K10))["questions"]
t0 = time.time()
with open(TRACE_OUT, "a") as f:
    for item in qs:
        if item["id"] in done:
            print(f"{item['id']} ya hecha, salto", flush=True)
            continue
        print(f"{item['id']} L{item['level']}: {item['question']}", flush=True)
        rec = run_question(item)
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        print(f"{item['id']} juez_ok={rec['judge']['ok']} "
              f"intentos_juez={rec['judge']['attempts']} "
              f"seleccionados={len(rec['selected_pages'])}/10 "
              f"lector_intentos={rec['reader']['attempts']} "
              f"-> {rec['final_answer'][:160]}", flush=True)
print(f"listo en {time.time() - t0:.1f}s -> {TRACE_OUT}")

## 3. Puntuar con el mismo scorer (con diagnóstico del juez)

Corre `agentic_rag/score_judge.py` con `--blank-modes`: escribe `results_agentic_judge_colab.md` con el puntaje, la comparación con los demás sistemas y el diagnóstico del juez (precisión/recall de selección contra las páginas de evidencia). Los modos de fallo quedan en PENDIENTE: se verifican a mano contra el corpus y se llenan en VERIFIED_MODES de `score_judge.py`, y se vuelve a correr sin `--blank-modes`.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "score_judge.py", "--trace", TRACE_OUT,
                    "--out", REPO + "/agentic_rag/results_agentic_judge_colab.md",
                    "--model-label", f"Ollama {JUDGE_OLLAMA_MODEL} (Colab T4)",
                    "--system-name", "RAG agentico juez Ollama (Colab)",
                    "--blank-modes"],
                   cwd=REPO + "/agentic_rag")
print("exit:", r.returncode)

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

El puntaje del juez es EL resultado agéntico que va al post (el ReAct 2/12 queda como negativo documentado). Sin esta traza, el draft no se puede actualizar con el resultado real.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('rag_baseline/retrieval_hybrid_k10.json', 'Retrieval híbrido top-10 (12 preguntas)', True), ('agentic_rag/run_trace_judge_colab.jsonl', 'Traza juez + lector (12 preguntas)', True), ('agentic_rag/results_agentic_judge_colab.md', 'Resultados RAG agéntico juez', True)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)